# Driven ML pipeline

Driven runs on a dashboard-mounted phone. MediaPipe Face Landmarker, a pre-trained network, turns
each camera frame into eye, mouth and head-pose signals on the phone. Driven's own model is a small
head on top: it scores each 60-second window for **drowsiness**, compared with the driver's own alert
baseline, and drives a spoken in-car alert and texts to parents who opted in. **Eyes off the road**
for more than 2 seconds is a fixed rule (NHTSA visual-manual guidelines, 78 FR 24818), not a model.

Driven makes **no intoxication claim**. No licensed, labeled video of impaired drivers exists, nobody
may drive impaired to create it (Tex. Penal Code 49.04), and advertising the claim without evidence
violates FTC Act section 5. The model card says so in plain words, and the compliance gate fails any
run whose user-facing text claims it.

This notebook documents each environment: dev, train, validate, test, staging and production.

## 1. Colab setup

In Colab, put the repository on Google Drive (unzipped, or as `DRIVEN-CONRAD-MLPRODUCT.zip` in
`MyDrive`), and add `KAGGLE_USERNAME`, `KAGGLE_KEY` and `HF_TOKEN` under **Secrets** if you want the
dataset search. Outside Colab this cell does nothing: install with
`.venv/bin/python -m pip install -e "ml[dev]"` instead.

In [1]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    repo = Path("/content/drive/MyDrive/DRIVEN-CONRAD-MLPRODUCT")
    archive = repo.with_suffix(".zip")
    if not repo.is_dir() and archive.is_file():
        with zipfile.ZipFile(archive) as bundle:
            bundle.extractall(repo.parent)
    install = [sys.executable, "-m", "pip", "install", "-q", "-e", f"{repo}/ml[discovery]"]
    subprocess.run(install, check=True)
    sys.path.insert(0, str(repo / "ml"))
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"):
        try:
            os.environ[name] = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
            print(f"Colab secret {name} not set; skipping it")
print("Colab" if IN_COLAB else "Local run: nothing to set up")

Local run: nothing to set up


## 2. Environments

Each environment runs a fixed list of stages, so none can skip the compliance gate or promote from
the wrong place. Dev runs everything except promotion, on synthetic people. Train, validate, test and
staging run on real, licensed video. Production only promotes, and only with a named approver.

In [2]:
import logging

import pandas as pd

from driven_ml.settings import STAGES_BY_ENVIRONMENT

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 200)
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s: %(message)s", force=True)

pd.DataFrame(
    [{"environment": env.value, "stages": " -> ".join(stages)} for env, stages in STAGES_BY_ENVIRONMENT.items()]
)

,environment,stages
0,dev,compliance -> discover -> features -> windows -> split -> train -> validate -> test -> package -> export_app_fixtures -> replay -> economics -> stage_gate
1,train,compliance -> features -> windows -> split -> train
2,validate,compliance -> windows -> split -> validate
3,test,compliance -> windows -> split -> test
4,staging,compliance -> windows -> split -> package -> export_app_fixtures -> replay -> economics -> stage_gate
5,production,compliance -> promote


## 3. Compliance

The gate runs first in every environment. It checks the legal flags in `product.json`, scans the
landing page, app and backend (when they sit beside this pipeline) for banned claims and forbidden
APIs, and fails the run on any finding. The rules are engineering guardrails derived from the team's
legal review, not legal advice; a lawyer should review them before launch.

In [3]:
from dataclasses import asdict

from driven_ml.compliance import GATED_SETTINGS, LEGAL_RULES, enforce_compliance
from driven_ml.product import load_product
from driven_ml.settings import find_repo_root

repo_root = find_repo_root()
product = load_product(repo_root)
enforce_compliance(product, repo_root)

rules = pd.DataFrame([asdict(rule) for rule in LEGAL_RULES])
gated = pd.DataFrame(
    [
        {**asdict(gate), "allowed_without_opinion": ", ".join(sorted(gate.allowed_without_opinion))}
        for gate in GATED_SETTINGS
    ]
)
display(rules)
display(gated)

INFO    driven_ml.compliance.checks: compliance gate passed: 7 product flags and 1 gated setting checked, repository scan clean


,rule_id,description,legal_basis,flag,required_value
0,L1,"No automated, artificial or prerecorded voice calls to 911 or any emergency line",47 CFR 64.1200(a)(1)(i),auto_call_911,False
1,L2,Automated texts only to contacts with recorded consent,47 CFR 64.1200 (prior express consent),require_contact_consent,True
2,L3,"Nothing for the driver to tap, read or say while moving; alerts are audio and clear by themselves",Tex. Transp. Code 545.4251; 545.424,driver_input_while_moving,False
3,L4,No cabin audio recording or voice recognition (voiceprints are biometric identifiers),Tex. Bus. & Com. Code 503.001,record_audio,False
4,L5,"Notice and consent from the driver, and a guardian for minors, before measuring a face",Tex. Bus. & Com. Code 503.001,require_biometric_consent,True
5,L6,Face data and personal baselines never leave the device,Tex. Bus. & Com. Code 503.001; TDPSA,upload_face_data,False
6,L7,Never sell or share personal or biometric data; partners see opt-in aggregates only,TDPSA (Tex. Bus. & Com. Code ch. 541); 503.001,sell_personal_data,False
7,L8,"No unsubstantiated claims ('detects drunk driving', 'measures BAC', 'prevents crashes')",FTC Act section 5,NaN,None
8,L9,Production models train only on commercially licensed data or the team's consented recordings,Dataset licenses; copyright,NaN,None
9,L10,"No scraping YouTube, social media or stock sites for training data",Platform terms of service,NaN,None


,rule_id,setting,allowed_without_opinion,opinion_flag,description,legal_basis
0,G1,tx_under18_device,"no_sim_device, partner_dashcam",legal_opinion_tx_under18,"Texas drivers under 18: no wireless device in use while driving, hands-free included",Tex. Transp. Code 545.424


The dataset registry records each dataset's license and which track may use it. The research track
may use anything not marked `never_use`; the commercial track, the only one that can ship, needs a
license that clearly allows commercial use (L9). Scraped and stock footage is never used (L10).

In [4]:
from driven_ml.data import registry_table

shown = ["key", "name", "license", "commercial_use", "research_track", "commercial_track", "intended_use"]
registry_table()[shown]

,key,name,license,commercial_use,research_track,commercial_track,intended_use
0,team_recordings,Driven parked-car recordings,"own, consented",True,True,True,"Replay tests, baseline tuning, commercial training"
1,hf_closed_open_eyes,closed-open-eyes,odc-by,True,True,True,Eye-state check of landmark features; verify image provenance first
2,uta_rldd,UTA Real-Life Drowsiness Dataset,research; cite CVPRW 2019,None,True,False,Research track: main drowsiness benchmark
3,kaggle_ddd,Driver Drowsiness Dataset (frames from UTA-RLDD),inherits UTA-RLDD,None,True,False,"Research track only; same people as UTA-RLDD, never mix across splits"
4,nitymed,NITYMED night-time drivers,free with citation; commercial use not stated,None,True,False,Night-time test set; ask the authors before commercial use
5,dmd,Driver Monitoring Dataset,CC BY-NC-ND 4.0,False,True,False,"Research track only (distraction, gaze zones)"
6,yawdd,YawDD,research use,False,True,False,Research track only (yawn detection)
7,ul_dd,UL-DD,"research use license, restricted",None,True,False,Research track if access is granted
8,state_farm,State Farm Distracted Driver,competition rules,False,False,False,Not for the product
9,dif,Dataset of Intoxicated Faces,scraped from YouTube,False,False,False,Not for any track (L10)


## 4. Dataset discovery

Discovery searches Hugging Face and Kaggle for drowsiness, eye-closure, yawn and driver-monitoring
datasets, and sorts the hits by license: commercially usable first, unknown next, restricted last
(`-nc`, `-nd` and research-only licenses). It writes `reports/discovered_datasets.csv` and never
downloads data or edits the registry; a person reads each license and data card before adding a
reviewed entry. It runs here only when Hugging Face or Kaggle credentials are present.

In [5]:
from driven_ml.data.discovery import run_discovery
from driven_ml.settings import RunConfig
from driven_ml.workspace import Workspace

has_credentials = (
    bool(os.environ.get("HF_TOKEN") or os.environ.get("KAGGLE_KEY"))
    or (Path.home() / ".kaggle" / "kaggle.json").is_file()
)
if has_credentials:
    discovery = run_discovery(Workspace(RunConfig.create("dev", run_discovery=True)).ensure())
    print(discovery.status_by_hub)
    display(pd.read_csv(discovery.report_path).head(20))
else:
    print("Discovery skipped: no HF_TOKEN, KAGGLE_KEY or ~/.kaggle/kaggle.json")

Discovery skipped: no HF_TOKEN, KAGGLE_KEY or ~/.kaggle/kaggle.json


## 5. Dev run

Dev runs every stage except promotion on 30 synthetic people, each with their own eye size, blink
habit and drowsiness visibility. It proves the whole pipeline works before any real video exists.

In [6]:
from driven_ml import run_pipeline

state = run_pipeline(RunConfig.create("dev", run_discovery=False))

INFO    driven_ml.pipeline: dev run on synthetic data, research track: compliance -> discover -> features -> windows -> split -> train -> validate -> test -> package -> export_app_fixtures -> replay -> economics -> stage_gate


INFO    driven_ml.pipeline: [1/13] compliance


INFO    driven_ml.compliance.checks: compliance gate passed: 7 product flags and 1 gated setting checked, repository scan clean


INFO    driven_ml.pipeline: [2/13] discover


INFO    driven_ml.pipeline: discover: skipped (--no-discovery)


INFO    driven_ml.pipeline: [3/13] features


... (35 outputs left out)

INFO    driven_ml.staging.gate: PASS replay: at least 90% of scripted events caught: 100% of 3 events caught, 0 false alerts (replayed sessions: 1); the synthetic scripted person is outside training


INFO    driven_ml.staging.gate: PASS parity fixtures reproduce (< 1e-6): max error 0.0e+00; 16 of 51 rows have missing values (at least 16 needed)


INFO    driven_ml.staging.gate: PASS unit economics: margin at least 60% at the test false-alert rate: family plan margin 88.3% at 0.00 false alerts/h (break-even 5.32/h)


INFO    driven_ml.staging.gate: staging PASSED


## 6. Frames and windows (feature spec v4)

Each frame holds 13 signals from the landmarks. Each 60-second window, every 15 seconds, summarizes
them in 12 features, and every one of them is chosen so that **where the phone is mounted cannot move
it**. A nearer camera makes every eye look more open, and a tilted mount shifts every head angle, so
absolute levels such as mean eye opening or mean pitch are left out. Instead:

- eye closure is measured against the eye's own open level in the window (its 90th-percentile EAR),
  and PERCLOS is the share of time the eye is at least 80% closed: the P80 measure of Wierwille et al.,
  NHTSA DOT HS 808 247 (1994), validated in Dinges and Grace, FHWA-MCRT-98-006 (1998);
- blinks and microsleeps are runs of at least 50% closure;
- head nods are dips below the window's own median pitch, and gaze-off is measured from the driver's
  own centre, not the camera's axis;
- the spread of pitch and yaw, yawns (a ratio, so distance cancels) and face coverage complete it.

Each later window also carries its `delta_` from the driver's baseline (section 7).

In [7]:
frames = pd.read_parquet(state.workspace.frames_path)
display(frames.head())

windows = pd.read_parquet(state.workspace.windows_path)
person = windows["subject"].iloc[0]
columns = ["session", "t0", "perclos", "delta_perclos", "closure_mean", "delta_closure_mean", "head_nods"]
windows.loc[windows["subject"] == person, columns].groupby("session").head(4)

,subject,session,clip,label,t,ear_left,ear_right,ear,mar,yaw,pitch,roll,gaze_horizontal,blink_left,blink_right,jaw_open,look_down,face
0,S000,S000_L0,drive,0,0.0,0.327390,0.333196,0.332188,0.099131,8.899772,0.500746,1.075663,0.079277,0.043758,0.043758,0.000000,0.000000,1.0
1,S000,S000_L0,drive,0,0.1,0.345952,0.346171,0.346911,0.081866,0.700599,-1.546657,0.155255,0.037225,0.001376,0.001376,0.000000,0.038666,1.0
2,S000,S000_L0,drive,0,0.2,0.362962,0.354044,0.358748,0.091422,5.676358,-0.593435,3.089660,-0.003110,0.000000,0.000000,0.000000,0.014836,1.0
3,S000,S000_L0,drive,0,0.3,0.325397,0.316596,0.325428,0.092255,7.903691,-0.165456,0.315191,-0.060300,0.063215,0.063215,0.000000,0.004136,1.0
4,S000,S000_L0,drive,0,0.4,0.342491,0.349104,0.339913,0.101434,1.585566,-0.543448,2.579110,-0.018449,0.021521,0.021521,0.001434,0.013586,1.0


,session,t0,perclos,delta_perclos,closure_mean,delta_closure_mean,head_nods
0,S000_L0,75.0,0.016779,-0.003131,0.084660,-0.015526,0
1,S000_L0,90.0,0.021922,0.002013,0.098987,-0.001199,0
2,S000_L0,105.0,0.027119,0.007209,0.102998,0.002812,0
3,S000_L0,120.0,0.037415,0.017506,0.112577,0.012391,0
32,S000_L1,0.0,0.040886,0.020977,0.125724,0.025537,0
33,S000_L1,15.0,0.032423,0.012514,0.117842,0.017655,0
34,S000_L1,30.0,0.032423,0.012514,0.115244,0.015058,0
35,S000_L1,45.0,0.037415,0.017506,0.132387,0.032201,0
69,S000_L2,0.0,0.020305,0.000395,0.116642,0.016456,2
70,S000_L2,15.0,0.008518,-0.011391,0.115722,0.015536,2


## 7. Two baseline scopes

The baseline is the driver's own alert face, like Face ID setup, and its scope depends on the data:

- **person** (public datasets such as UTA-RLDD): the first two minutes of the person's alert video are
  the baseline for all their videos. Every video is a separate recording, so a camera change between
  videos can still pass for drowsiness, and this scope's numbers are reported but never trusted alone.
- **session** (team recordings, `ml/RECORDING_PROTOCOL.md`): a 2-minute calibration clip and the drive
  are recorded in one mount and one sitting, as the phone does. The calibration windows are the
  baseline for that session's drive and never become training windows. Drive windows are labeled from
  a script written live: drowsy when they overlap an acted drowsiness, yawn or nod; alert when at
  least 30 s from every event; left out otherwise, including looking away, which is a fixed rule.

Below, one synthetic team session shows the labels and the calibration baseline.

In [8]:
from driven_ml.features.synthetic import simulate_team_session
from driven_ml.windows import add_baseline_deltas, label_scripted_windows, slide_windows

team_frames, team_script = simulate_team_session("team_recordings:T00", "s1", seed=3)
team_session = "team_recordings:T00:s1"
team_windows = label_scripted_windows(slide_windows(team_frames), team_script.assign(session=team_session))
display(team_script)
display(team_windows.groupby(["clip", "label"]).size().rename("windows").reset_index())
scoped = add_baseline_deltas(team_windows)
scoped[["clip", "baseline_scope", "t0", "label", "perclos", "delta_perclos"]].head(8)

,start_s,end_s,event
0,150.0,190.0,acted_drowsiness
1,330.0,340.0,acted_yawn
2,480.0,485.0,acted_nod
3,600.0,603.0,acted_eyes_off
4,720.0,760.0,acted_drowsiness


,clip,label,windows
0,calibration,0,9
1,drive,0,13
2,drive,2,20


,clip,baseline_scope,t0,label,perclos,delta_perclos
0,drive,session,0.0,0,0.0,-0.00229
1,drive,session,15.0,0,0.0,-0.00229
2,drive,session,30.0,0,0.0,-0.00229
3,drive,session,45.0,0,0.0,-0.00229
4,drive,session,60.0,0,0.0,-0.00229
5,drive,session,105.0,2,0.0,-0.00229
6,drive,session,120.0,2,0.0,-0.00229
7,drive,session,135.0,2,0.0,-0.00229


## 8. Validation

Every candidate is scored with 5-fold cross-validation grouped by person, so no number comes from a
person the model trained on. The claim being tested is the personal baseline: the Driven models use
the window features plus their deltas from the driver's baseline, B3 is the same logistic regression
without the deltas, and B1 is a fixed PERCLOS threshold for everyone. If B3 matches Driven, the
baseline is not earning its keep on this data, and that is a finding to report, not hide. Thresholds
are chosen by the false-alert budget (0.2 per hour), and probabilities are temperature-calibrated on
out-of-fold logits.

In [9]:
validation = state.validation.to_frame()
display(validation.round(3))
display(state.validation.scope_frame().round(3))
best = state.validation.best()
print(f"Exported: {best.name} ({len(best.inputs)} inputs), threshold {best.threshold:.2f}")

,model,inputs,meets_budget,auc,balanced_accuracy,false_alerts_per_hour,drowsy_sessions_caught,temperature,threshold
0,Driven logistic regression,24,True,1.000,1.000,0.000,1.000,0.373,0.050
1,Driven MLP,24,True,1.000,0.998,0.000,1.000,0.713,0.110
2,B3 no personal baseline,12,True,1.000,1.000,0.000,1.000,0.530,0.050
3,B1 fixed PERCLOS threshold,1,None,0.645,0.612,12.955,0.955,NaN,0.035


,model,baseline_scope,people,windows,auc,balanced_accuracy,false_alerts_per_hour,drowsy_sessions_caught
0,Driven logistic regression,person,22,1518,1.000,1.000,0.000,1.000
1,Driven MLP,person,22,1518,1.000,0.998,0.000,1.000
2,B3 no personal baseline,person,22,1518,1.000,1.000,0.000,1.000
3,B1 fixed PERCLOS threshold,person,22,1518,0.645,0.612,12.955,0.955


Exported: Driven logistic regression (24 inputs), threshold 0.05


## 9. Locked test

Test people are chosen by a hash of their id, so nobody can reshuffle them, and only the test stage
touches them. Every test run is logged, but the first result for a model hash is the one kept:
rerunning cannot swap a bad number for a lucky one. The report breaks every metric down by baseline
scope and gives the within-recording AUC over held-out team sessions (none in dev).

In [10]:
report = state.test_report
print(f"{report.people} test people, {report.windows} windows, prior runs of this model: {report.prior_runs}")
overall = {"Driven": asdict(report.driven), "B1 fixed PERCLOS": asdict(report.fixed_threshold)}
display(pd.DataFrame(overall).T.round(3))
display(pd.DataFrame([asdict(score) for score in report.by_scope]).round(3))
print(
    f"within-recording AUC {report.within_recording.auc:.3f} "
    f"over {report.within_recording.sessions} held-out team sessions"
)

8 test people, 552 windows, prior runs of this model: 0


,auc,balanced_accuracy,false_alerts_per_hour,drowsy_sessions_caught
Driven,1.000,0.990,0.000,1.00
B1 fixed PERCLOS,0.573,0.541,13.125,0.75


,model,baseline_scope,people,windows,auc,balanced_accuracy,false_alerts_per_hour,drowsy_sessions_caught
0,Driven,person,8,552,1.000,0.990,0.000,1.00
1,B1 fixed PERCLOS,person,8,552,0.573,0.541,13.125,0.75


within-recording AUC nan over 0 held-out team sessions


## 10. Staging

Staging packages the model, its model card, the feature spec and the parity fixtures, replays scripted
sessions exactly as the phone would score them, checks the unit economics at the measured false-alert
rate, and writes a manifest. Every check must pass before anyone may promote. Three checks exist
because of the camera confound (section 11):

- **null control**: the pipeline must score at chance when the labels carry no drowsiness;
- **within-recording AUC ≥ 0.80** on held-out team sessions, gated on real data and only reported in dev;
- **replay** on people the model never trained on: `replay_only` people and locked test people. Real
  data needs at least 2 replay people and 6 acted drowsiness events. The dev scripted person is
  synthetic and outside training.

The parity fixtures keep missing values as JSON `null`: 15 rows have 1 to 5 inputs removed and one row
has none at all, so a phone port that skips `fill_values` fails parity.

In [11]:
manifest = state.manifest
display(pd.DataFrame([asdict(check) for check in manifest.checks]))
print("staging", "PASSED" if manifest.passed else "FAILED")
display(pd.DataFrame([asdict(result) for result in state.replay]))

,name,passed,detail
0,compliance: product flags,True,L1-L7 and G1 satisfied
1,compliance: no banned claims in the model card (L8),True,no banned claims
2,datasets allowed for the track (L9),True,"skipped: synthetic data has no dataset licenses, and it can never ship"
3,beats the fixed-threshold baseline on test,True,balanced accuracy 0.990 vs B1 0.541
4,null control: pipeline scores at chance when labels carry no signal,True,"AUC seed 101: 0.556, seed 102: 0.480, seed 103: 0.521; mean 0.519 (limits: mean <= 0.6, each <= 0.65)"
5,within-recording AUC on held-out team sessions >= 0.80,True,not gated in dev; scripted session within-recording AUC 1.000
6,false alerts at most 0.2/h on test,True,0.00 false alerts per hour over 8 test people
7,replay: at least 90% of scripted events caught,True,"100% of 3 events caught, 0 false alerts (replayed sessions: 1); the synthetic scripted person is outside training"
8,parity fixtures reproduce (< 1e-6),True,max error 0.0e+00; 16 of 51 rows have missing values (at least 16 needed)
9,unit economics: margin at least 60% at the test false-alert rate,True,family plan margin 88.3% at 0.00 false alerts/h (break-even 5.32/h)


staging PASSED


,session,person,group,events,detected,false_alerts,alert_times,within_recording
0,synthetic/SCRIPT_1,SCRIPT,replay,3,3,0,"(255.0, 405.0, 555.0)","{'sessions': 1, 'pairs': 270, 'concordant': 270.0}"


In [12]:
import json

import numpy as np

from driven_ml.modeling import ModelArtifact, predict_probability

stage_dir = state.workspace.staging_dir(manifest.model_sha256[:12])
staged = ModelArtifact.from_json(json.loads((stage_dir / "model.json").read_text()))
parity = json.loads((stage_dir / "parity_fixtures.json").read_text())
parity_inputs = np.array(parity["inputs"], dtype=float)
expected = np.array(parity["expected"])
error = np.max(np.abs(predict_probability(staged, parity_inputs) - expected))
print(f"parity: {len(parity_inputs)} rows, {parity['rows_with_missing']} with missing values")
print(f"max error {error:.1e} (tolerance {parity['tolerance']})")
skipped = np.max(np.abs(predict_probability(staged, np.nan_to_num(parity_inputs)) - expected))
print(f"a port that writes 0 instead of the fill values misses by {skipped:.3f}")
print((stage_dir / "MODEL_CARD.md").read_text())

parity: 51 rows, 16 with missing values
max error 0.0e+00 (tolerance 1e-06)
a port that writes 0 instead of the fill values misses by 0.044
**What it does.** Scores 60-second windows of eye, head and mouth signals for signs of drowsiness,
compared with the driver's own alert baseline. It drives the in-car spoken alert and parent texts.

**What it does not do.** It does not measure alcohol or any other substance, does not identify
intoxication, and is not a medical device. It never replaces the rule of not driving after drinking.

**Data.** Track: research. Datasets: synthetic.

**Results on 8 held-out people.**

| Model | AUC | Balanced accuracy | False in-car alerts per hour | Drowsy sessions caught |
|---|---|---|---|---|
| Driven | 1.000 | 0.990 | 0.00 | 100% |
| Fixed PERCLOS threshold (B1) | 0.573 | 0.541 | 13.12 | 75% |

**Replay.** Scripted, acted sessions are replayed for people the model never trained on.

**Limits.** Phone cameras lose eye detail in the dark and behind sungla

## 11. What would fool this model, and how we check

**The camera, not the driver.** On real data every video has its own mount angle and distance. A model
can learn "this is a different recording" instead of "this driver is drowsy". That is shortcut
learning (Geirhos et al., "Shortcut learning in deep neural networks," *Nature Machine Intelligence*
2, 665-673, 2020). In UTA-RLDD every video is a separate recording and the drowsy videos are drowsy
from the start (Ghoddoosian, Alexander and Ross, CVPRW 2019), so its baseline always comes from a
different recording than its drowsy data.

**Defence 1: dev data contains the confound.** Every synthetic recording draws its own camera
placement, which scales eye and mouth opening and offsets head pose and gaze.

**Defence 2: camera-invariant features (spec v4).** Section 6.

**Defence 3: the null-signal control.** Forty synthetic people whose drowsy sessions show no
drowsiness at all go through the whole pipeline, three seeds. A sound pipeline scores at chance; the
gate needs a mean AUC of at most 0.60 and every seed at most 0.65. This is a permutation-style check
(Ojala and Garriga, "Permutation tests for studying classifier performance," *JMLR* 11, 1833-1863,
2010). On feature spec v3 it scored AUC 0.516, 0.779 and 0.972 (mean 0.756): the confound, measured.
On v4 it scores:

In [13]:
null_control = pd.DataFrame(manifest.null_control["results"])
display(null_control)
print(f"mean AUC {manifest.null_control['mean_auc']:.3f}, passed: {manifest.null_control['passed']}")

,seed,auc,chosen_model,test_people
0,101,0.556081,B3 no personal baseline,10
1,102,0.479890,B3 no personal baseline,10
2,103,0.520566,Driven MLP,10


mean AUC 0.519, passed: True


**Defence 4: accuracy inside one recording.** On real data the number that counts is the AUC between
drowsy and alert windows of the *same* drive clip, pooled over held-out team sessions. No camera
difference can inflate it, and a model cannot ship without it (at least 0.80).

## 12. Production

Promotion is refused for this model, and the reasons say why: it was trained on synthetic data on the
research track. The real path is:

1. Record consented team sessions per `ml/RECORDING_PROTOCOL.md`: a calibration clip and a scripted
   drive in one mount, in a parked car with acted cues only (L11), with `consent.csv` filled in for
   every person, plus a guardian's consent for minors (L5). Mark at least two people `replay_only`.
2. Train on the commercial track: `--env train --track commercial`.
3. Run `test`, then `staging`, until every staging check passes.
4. Promote with a named approver: `--env production --track commercial --approved-by "<name>"`.

In [14]:
from driven_ml.errors import ComplianceError
from driven_ml.release import promote

attempt = RunConfig.create("dev", run_discovery=False, approved_by="Notebook reviewer")
try:
    promote(attempt, state.workspace, state.product)
except ComplianceError as refused:
    print(refused)

Promotion refused:
- only commercial-track models can ship (L9)
- models trained on synthetic data can never ship
- datasets not cleared for commercial use (L9): synthetic


## 13. Running each environment

From the repository, or from Colab after the setup cell (`!cd <repo>/ml && python -m driven_ml ...`):

| Environment | Command |
|---|---|
| dev | `python -m driven_ml --env dev --no-discovery` |
| train | `python -m driven_ml --env train --track commercial` (add `--reuse-features` to skip re-extraction) |
| validate | `python -m driven_ml --env validate --track commercial` |
| test | `python -m driven_ml --env test --track commercial` |
| staging | `python -m driven_ml --env staging --track commercial` |
| production | `python -m driven_ml --env production --track commercial --approved-by "<name>"` |

Public video goes in `driven_work/raw/<dataset>/<person>/{0,5,10}.mp4` (UTA-RLDD naming: 0 alert,
5 low vigilance, 10 drowsy); team sessions go in `driven_work/recordings/<person>/<session>/` with
`calibration.mp4`, `drive.mp4` and `script.csv`. Add
`--write-app --app-dir <app>` to copy fixtures or a released model into the app.